In [1]:
#!pip install selenium pandas
#!pip install html5lib

In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    ElementClickInterceptedException,
    TimeoutException,
)

def scrape_understat(league: str, year: int):
    """Scrape Understat league data (season, players, matches)."""

    # Setup browser
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_experimental_option('excludeSwitches', ['enable-logging'])
    driver = webdriver.Chrome(options=options)

    # Go to league page
    url = f"https://understat.com/league/{league}/{year}"
    print(f"Navigating to {url}")
    driver.get(url)

    # Wait for user actions
    print("Waiting 30 seconds for manual changes...")
    time.sleep(30)
    print("Starting extraction...")

    wait = WebDriverWait(driver, 20)

    # --- Helper: convert table HTML → DataFrame ---
    def table_to_df(container_element):
        """Parse table in a container into DataFrame."""
        try:
            table = container_element.find_element(By.TAG_NAME, "table")
        except NoSuchElementException:
            table = container_element  # Fallback
        
        rows = table.find_elements(By.TAG_NAME, "tr")
        data = []
        for row in rows:
            # Try header first, else normal cells
            cells = row.find_elements(By.TAG_NAME, "th") or row.find_elements(By.TAG_NAME, "td")
            cell_texts = [c.text.strip() for c in cells]
            if cell_texts:
                data.append(cell_texts)
        
        # Skip empty or invalid tables
        if len(data) < 2:
            return pd.DataFrame()
        return pd.DataFrame(data[1:], columns=data[0])

    # --- 1️⃣ SEASON TABLE ---
    print("Extracting season table...")
    try:
        all_tables = driver.find_elements(By.TAG_NAME, "table")
        season_table = pd.DataFrame()

        for table in all_tables:
            rows = table.find_elements(By.TAG_NAME, "tr")
            if len(rows) > 15:
                first_row_text = rows[0].text.lower() if rows else ""
                if any(k in first_row_text for k in ['team', 'squad', 'pts', 'points', 'position']):
                    # Scroll to view
                    driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", table)
                    time.sleep(1.5)
                    season_table = table_to_df(table)
                    break
        
        if season_table.empty:
            season_table = pd.DataFrame()
    except Exception:
        season_table = pd.DataFrame()
    print(f"✓ Season table: {season_table.shape[0]} rows")

    # --- 2️⃣ PLAYERS TABLE ---
    print("Extracting players table...")
    players_xpath = "/html/body/div[1]/div[3]/div[4]/div"
    try:
        # Scroll to players table
        el = driver.find_element(By.XPATH, players_xpath)
        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", el)
        time.sleep(1.5)
        wait.until(EC.presence_of_element_located((By.XPATH, players_xpath)))
        players_table = table_to_df(el)
    except Exception:
        players_table = pd.DataFrame()
    print(f"✓ Players table: {players_table.shape[0]} rows")

    # --- 3️⃣ MATCHES (loop through matchdays) ---
    print("Extracting matches across all matchdays...")
    matches_xpath_root = "/html/body/div[1]/div[3]/div[2]/div/div/div[2]"
    prev_button_xpath = "/html/body/div[1]/div[3]/div[2]/div/div/button[1]"

    # Scroll to navigation buttons
    driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});",
                         driver.find_element(By.XPATH, prev_button_xpath))
    time.sleep(1)

    all_matchday_dfs = []

    # --- Extract one matchday ---
    def extract_matchday():
        """Grab all matches from current matchday."""
        try:
            container = driver.find_element(By.XPATH, matches_xpath_root)
            date_containers = container.find_elements(By.XPATH, "./div")
        except:
            return pd.DataFrame()
    
        if not date_containers:
            return pd.DataFrame()
    
        data = []
        for date_container in date_containers:
            try:
                # Get match date
                date_elem = date_container.find_element(By.XPATH, "./div[1]")
                match_date = date_elem.text.strip()

                # Get all matches under this date
                matches_container = date_container.find_element(By.XPATH, "./div[2]")
                individual_matches = matches_container.find_elements(By.XPATH, "./div")
                
                for match in individual_matches:
                    try:
                        match_text = match.text.strip()
                        if not match_text:
                            continue
                        lines = [line.strip() for line in match_text.split('\n') if line.strip()]

                        # Try extracting by CSS
                        try:
                            home_elem = match.find_element(By.CSS_SELECTOR, ".team-home .team-title")
                            away_elem = match.find_element(By.CSS_SELECTOR, ".team-away .team-title")
                            score_elem = match.find_element(By.CSS_SELECTOR, ".match-info .score")
                            home_team = home_elem.text.strip()
                            away_team = away_elem.text.strip()
                            score = score_elem.text.strip()
                        except:
                            # Fallback to manual text parse
                            home_team = lines[0]
                            away_team = lines[-1]
                            score = ""
                            for line in lines[1:-1]:
                                if len(line) <= 3 and any(c.isdigit() for c in line):
                                    score = line
                                    break
                        
                        # Add match info
                        if home_team and away_team:
                            data.append({
                                "date": match_date,
                                "home_team": home_team,
                                "away_team": away_team,
                                "score": score
                            })
                    except Exception:
                        continue
            except Exception:
                continue
    
        return pd.DataFrame(data)

    # --- Loop through all matchdays ---
    while True:
        df = extract_matchday()
        if not df.empty:
            all_matchday_dfs.insert(0, df)

        try:
            prev_button = driver.find_element(By.XPATH, prev_button_xpath)
            disabled = driver.execute_script("return arguments[0].disabled;", prev_button)
            if disabled:
                break

            # Go to previous matchday
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", prev_button)
            time.sleep(0.5)
            try:
                prev_button.click()
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", prev_button)
            time.sleep(3)
        except (NoSuchElementException, TimeoutException):
            break

    # Combine all matchdays
    matches_table = pd.concat(all_matchday_dfs, ignore_index=True) if all_matchday_dfs else pd.DataFrame()
    print(f"✓ Matches table: {matches_table.shape[0]} matches from {len(all_matchday_dfs)} matchdays")

    # Close browser
    driver.quit()
    print("Extraction complete!\n")

    return season_table, players_table, matches_table

In [44]:
season, players, matches = scrape_understat("EPL", 2023)

Navigating to https://understat.com/league/EPL/2023
Waiting 30 seconds for manual changes...
Starting extraction...
Extracting season table...
✓ Season table: 21 rows
Extracting players table...
✓ Players table: 12 rows
Extracting matches across all matchdays...
✓ Matches table: 380 matches from 37 matchdays
Extraction complete!



In [45]:
season

,№,Team,M,W,D,L,G,GA,PTS,xG,NPxG,xGA,NPxGA,NPxGD,PPDA,OPPDA,DC,ODC,xPTS
0,№,Team,M,W,D,L,G,GA,PTS,xG,NPxG,xGA,NPxGA,NPxGD,PPDA,OPPDA,DC,ODC,xPTS
1,1,Manchester City,38,28,7,3,96,34,91,89.55-6.45,81.94,37.27+3.27,34.99,+46.95,11.25,20.09,516,155,83.07-7.93
2,2,Arsenal,38,28,5,5,91,29,89,84.39-6.61,76.78,31.78+2.78,29.50,+47.28,9.60,16.83,531,151,81.94-7.06
3,3,Liverpool,38,24,10,4,86,41,82,94.79+8.79,87.65,47.40+6.40,46.64,+41.01,7.66,16.64,510,233,76.88-5.12
4,4,Aston Villa,38,20,8,10,76,61,68,67.42-8.58,64.38,65.08+4.08,63.43,+0.95,13.68,10.31,344,268,55.43-12.57
5,5,Tottenham,38,20,6,12,74,61,66,73.35-0.65,71.83,68.07+7.07,62.74,+9.09,7.50,12.34,493,264,57.42-8.58
6,6,Chelsea,38,18,9,11,77,63,63,80.88+3.88,71.75,62.60-0.40,58.78,+12.97,9.57,14.18,382,292,64.20+1.20
7,7,Newcastle United,38,18,6,14,85,62,60,84.48-0.52,77.63,62.46+0.46,58.52,+19.10,10.26,11.06,348,274,65.59+5.59
8,8,Manchester United,38,18,6,14,57,58,60,60.33+3.33,54.99,74.76+16.76,69.43,-14.45,11.96,13.11,319,371,44.42-15.58
9,9,West Ham,38,14,10,14,60,74,52,54.53-5.47,50.72,77.91+3.91,70.30,-19.57,15.95,9.05,159,444,41.28-10.72


In [46]:
players

,№,Player,Team,Apps,Min,G,NPG,A,xG,NPxG,xA,xGChain,xGBuildup,xG90,NPxG90,xA90,xG90 + xA90,NPxG90 + xA90,xGChain90,xGBuildup90
0,№,Player,Team,Apps,Min,G,NPG,A,xG,NPxG,xA,xGChain,xGBuildup,xG90,NPxG90,xA90,xG90 + xA90,,,
1,1,Erling Haaland,Manchester City,31,2581,27,20,5,31.65+4.65,25.56+5.56,4.75-0.25,30.20,3.13,1.10,0.89,0.17,1.27,1.06,1.05,0.11
2,2,Cole Palmer,"Chelsea, Manchester City",34,2640,22,13,11,17.83-4.17,10.98-2.02,11.87+0.87,31.04,15.30,0.61,0.37,0.40,1.01,0.78,1.06,0.52
3,3,Alexander Isak,Newcastle United,30,2305,21,16,2,22.07+1.07,17.51+1.51,3.65+1.65,24.04,5.75,0.86,0.68,0.14,1.00,0.83,0.94,0.22
4,4,Dominic Solanke,Bournemouth,38,3346,19,17,3,21.41+2.41,19.12+2.12,3.54+0.54,22.59,4.08,0.58,0.51,0.10,0.67,0.61,0.61,0.11
5,5,Phil Foden,Manchester City,35,2895,19,19,8,11.31-7.69,11.31-7.69,8.52+0.52,30.42,17.00,0.35,0.35,0.26,0.62,0.62,0.95,0.53
6,6,Ollie Watkins,Aston Villa,37,3250,19,19,13,19.26+0.26,19.26+0.26,7.75-5.25,29.23,5.90,0.53,0.53,0.21,0.75,0.75,0.81,0.16
7,7,Mohamed Salah,Liverpool,32,2556,18,13,10,21.94+3.94,16.61+3.61,12.33+2.33,31.88,8.36,0.77,0.58,0.43,1.21,1.02,1.12,0.29
8,8,Son Heung-Min,Tottenham,35,2989,17,15,10,12.98-4.02,11.46-3.54,13.34+3.34,30.12,10.45,0.39,0.34,0.40,0.79,0.75,0.91,0.31
9,9,Jarrod Bowen,West Ham,34,3026,16,16,6,13.50-2.50,13.50-2.50,6.44+0.44,17.77,2.46,0.40,0.40,0.19,0.59,0.59,0.53,0.07


In [47]:
matches

,date,home_team,away_team,score
0,"Friday, August 11, 2023",Burnley,Manchester City,03
1,"Saturday, August 12, 2023",Arsenal,Nottingham Forest,21
2,"Saturday, August 12, 2023",Bournemouth,West Ham,11
3,"Saturday, August 12, 2023",Brighton,Luton,41
4,"Saturday, August 12, 2023",Everton,Fulham,01
...,...,...,...,...
375,"Sunday, May 19, 2024",Crystal Palace,Aston Villa,50
376,"Sunday, May 19, 2024",Liverpool,Wolverhampton Wanderers,20
377,"Sunday, May 19, 2024",Luton,Fulham,24
378,"Sunday, May 19, 2024",Manchester City,West Ham,31
